# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbhutto256/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Paper reviewed:** *The State of AI-Driven SEO — FlyRank Data Report, April 2026*  
Public source: https://state-of-seo-2026.flyrank.ai/

I am treating the paper as a useful example of public-facing applied research, not as something to “grade.” The questions below are the same questions I want someone to ask about my own work.

### Finding 1 — growing pages were younger than declining pages

The paper reports that growing pages averaged about **185 days old** and declining pages about **228 days old**, while average word count was almost the same in the two groups. I read this as an observed association in the paper's analysis, not evidence that age itself causes growth or decline.

**Methodology question I would ask:**  
**Where exactly does the growing/declining label come from, and is the age measurement strictly available before the period used to assign that label?** If the label and the page-age/freshness measurement use overlapping time information, the comparison could partly describe the way the groups were constructed rather than a forward-looking signal. I would also want to know how the result changes after controlling or stratifying for page type, starting visibility, and brand/client, because older pages may differ from younger pages in several ways besides age.

**What would make the claim stronger:** a clearly stated label window, a pre-label feature snapshot, and a grouped or time-aware comparison showing that the age gap remains directionally similar outside the exact pages used to discover it.

### Finding 2 — prediction was stronger on seen brands than unseen brands

The paper reports roughly **89–90%** performance for new pages from brands represented in training and about **75%** on brands the model had not seen before.

**Methodology question I would ask:**  
**Does the validation design fully separate brands/clients between training and the “unseen brand” evaluation, and what target/metric is the percentage describing?** A random page split can look strong if pages from the same brand appear in both train and test because the model can learn brand-specific patterns. A truly unseen-brand claim needs the split to happen at the brand/client level before preprocessing or model fitting.

I would also ask for the base rate and the distribution across repeated grouped splits. A single 75% result can be useful, but repeated client-grouped validation would show whether the measured generalization is stable or driven by a few easier/harder held-out brands.

**What I take from both findings:** the paper's public numbers are most useful when I can trace the label, timeline, split unit, base rate, and metric. I apply those same checks to my Week-5 model below.


In [7]:
# Paper findings recorded as structured evidence + shared notebook setup.
# The paper review itself is interpretive; the table simply keeps the public numbers explicit.

import os, getpass, duckdb, pandas as pd, numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score, confusion_matrix
)

paper_findings = pd.DataFrame([
    {
        "finding": "Growing vs declining page age",
        "paper_number": "185d growing vs 228d declining; word count ~1.5K in both",
        "methodology_focus": "label origin + feature/label timeline + confounding"
    },
    {
        "finding": "Growth prediction generalization",
        "paper_number": "~89–90% seen-brand pages vs ~75% unseen brands",
        "methodology_focus": "grouped validation + target/metric + base rate + stability"
    },
])

display(paper_findings)

PAPER_URL = "https://state-of-seo-2026.flyrank.ai/"
print("Paper reviewed:", PAPER_URL)


,finding,paper_number,methodology_focus
0,Growing vs declining page age,185d growing vs 228d declining; word count ~1....,label origin + feature/label timeline + confou...
1,Growth prediction generalization,~89–90% seen-brand pages vs ~75% unseen brands,grouped validation + target/metric + base rate...


Paper reviewed: https://state-of-seo-2026.flyrank.ai/


## 2. My model under an honest split (before/after)

My Week-5 model is **Logistic Regression** using five March 1–15 Google Search Console features to rank a binary `decline_proxy` defined from March 16–31 impressions.

The validation improvement I audit here is:

- **Before:** a naive stratified random row split. Pages from the same client can appear on both sides.
- **After:** `GroupShuffleSplit` by `client_hash_id`. No client can appear in both train and test.

This matters because pages belonging to one client can share site-level history, tracking patterns, editorial behavior, and search conditions. A random row split can reward the model for learning client-specific regularities that will not transfer cleanly to a new client.

I use the **same feature list, model pipeline, test fraction, random seed, and evaluation functions** for both versions. The held-out rows are not identical because the split rules are different, so I treat the before/after gap as a diagnostic of validation sensitivity rather than a perfectly controlled causal estimate of “how much leakage” existed.

My primary ranking metric remains **Precision@50** because this lane is a review-priority problem: if a team can inspect only 50 pages, how many of those 50 meet the decline proxy? I also report PR-AUC, ROC-AUC, balanced accuracy, precision, recall, and the test-set base rate so one number cannot hide the class balance.


In [8]:
# Rebuild the same Week-5 March frame, then compare naive random vs client-grouped validation.
# Add HF_TOKEN in Colab Secrets. The token is never printed or stored in the notebook.

%pip -q install duckdb huggingface_hub scikit-learn pandas numpy

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Enter Hugging Face READ token (hidden): ")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

FEATURE_COLS = [
    "first_half_impressions",
    "first_half_clicks",
    "first_half_avg_position",
    "first_half_active_days",
    "first_half_ctr_pct",
]
TARGET = "decline_proxy"
GROUP = "client_hash_id"

feature_sql = f"""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        NULLIF(gsc_avg_position, 0) AS gsc_avg_position
    FROM {DAILY}
    WHERE gsc_data_available IS TRUE
),
agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
        AVG(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 THEN gsc_avg_position END) AS first_half_avg_position,
        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
             AND gsc_impressions > 0 THEN report_date END
        ) AS first_half_active_days,
        SUM(CASE WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                 THEN gsc_impressions ELSE 0 END) AS second_half_impressions
    FROM daily
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    first_half_impressions,
    first_half_clicks,
    first_half_avg_position,
    first_half_active_days,
    100.0 * first_half_clicks / NULLIF(first_half_impressions, 0) AS first_half_ctr_pct,

    -- Audit-only future fields. These are NEVER included in FEATURE_COLS.
    second_half_impressions,
    100.0 * (second_half_impressions - first_half_impressions)
        / NULLIF(first_half_impressions, 0) AS future_trend_pct,

    CASE
        WHEN first_half_impressions > 0
         AND second_half_impressions < 0.80 * first_half_impressions
        THEN 1 ELSE 0
    END AS decline_proxy
FROM agg
WHERE first_half_impressions > 0
"""

model_df = con.sql(feature_sql).df()

print(f"Rows: {len(model_df):,}")
print(f"Clients: {model_df[GROUP].nunique():,}")
print(f"Overall decline-proxy base rate: {model_df[TARGET].mean():.3f}")

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )),
    ])

def precision_at_k(y_true, score, k=50):
    y_true = np.asarray(y_true)
    score = np.asarray(score)
    k = min(k, len(y_true))
    if k == 0:
        return np.nan
    top = np.argsort(-score, kind="mergesort")[:k]
    return float(y_true[top].mean())

def evaluate(y_true, prob, pred):
    y_true = np.asarray(y_true)
    return {
        "base_rate": float(np.mean(y_true)),
        "precision_at_50": precision_at_k(y_true, prob, 50),
        "pr_auc": average_precision_score(y_true, prob),
        "roc_auc": roc_auc_score(y_true, prob),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
    }

X = model_df[FEATURE_COLS]
y = model_df[TARGET].astype(int)

# BEFORE: naive row-level split
rnd_train_idx, rnd_test_idx = train_test_split(
    np.arange(len(model_df)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = make_model()
random_model.fit(X.iloc[rnd_train_idx], y.iloc[rnd_train_idx])
random_prob = random_model.predict_proba(X.iloc[rnd_test_idx])[:, 1]
random_pred = (random_prob >= 0.50).astype(int)
random_metrics = evaluate(y.iloc[rnd_test_idx], random_prob, random_pred)

# AFTER: client-grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
grp_train_idx, grp_test_idx = next(
    gss.split(X, y, groups=model_df[GROUP])
)

train_clients = set(model_df.iloc[grp_train_idx][GROUP])
test_clients = set(model_df.iloc[grp_test_idx][GROUP])
assert train_clients.isdisjoint(test_clients), "Client leakage detected in grouped split."

group_model = make_model()
group_model.fit(X.iloc[grp_train_idx], y.iloc[grp_train_idx])
group_prob = group_model.predict_proba(X.iloc[grp_test_idx])[:, 1]
group_pred = (group_prob >= 0.50).astype(int)
group_metrics = evaluate(y.iloc[grp_test_idx], group_prob, group_pred)

comparison = pd.DataFrame(
    [random_metrics, group_metrics],
    index=["Naive random row split", "Client-grouped split"]
)

display(comparison.round(3))
print()
print(f"Grouped train clients: {len(train_clients)}")
print(f"Grouped test clients:  {len(test_clients)}")
print(f"Client overlap:        {len(train_clients & test_clients)}")

gap = random_metrics["precision_at_50"] - group_metrics["precision_at_50"]
print(f"\nPrecision@50 gap (random - grouped): {gap:+.3f}")
print(
    "Interpretation: this gap is a validation-sensitivity finding. "
    "I use the grouped result as the number I would defend for unseen-client decision-support."
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 151,981
Clients: 44
Overall decline-proxy base rate: 0.327


,base_rate,precision_at_50,pr_auc,roc_auc,balanced_accuracy,precision,recall
Naive random row split,0.327,0.62,0.413,0.597,0.565,0.397,0.492
Client-grouped split,0.373,0.34,0.395,0.549,0.528,0.397,0.575



Grouped train clients: 35
Grouped test clients:  9
Client overlap:        0

Precision@50 gap (random - grouped): +0.280
Interpretation: this gap is a validation-sensitivity finding. I use the grouped result as the number I would defend for unseen-client decision-support.


## 3. Leakage audit

I audit the final Week-5 features against three leakage paths from the repo guidance.

### A. Label-derived features

The target is created from the **second-half March impression change**. Therefore `second_half_impressions`, `future_trend_pct`, or any field derived from the March 16–31 outcome window would directly or indirectly reveal the answer. They are retained below only so I can prove that the audit catches them; they are not model inputs.

### B. Future / overlapping windows

Every production feature in `FEATURE_COLS` comes only from **March 1–15**. The target uses **March 16–31**. This creates a visible decision boundary:

`features: Mar 1–15  |  decision point: Mar 15  |  label: Mar 16–31`

That is the minimum timeline discipline required for a forward-looking decline proxy.

### C. ID / decision leakage

`client_hash_id` and `content_hash_id` are used only for grouping/context. They are never predictors. I also do not use an existing product score, refresh flag, trend direction, or decision field as a feature.

### Deliberate leakage sanity check

A good audit harness should react when I intentionally give the model the answer. I therefore train one **audit-only** grouped model with `future_trend_pct` as a feature. Because `decline_proxy` is defined from this future percentage, its score should become implausibly strong. That is not a model improvement; it is evidence that the leakage test is capable of detecting the failure mode.

### Real failure examples

Finally, I inspect the most confident false positives and false negatives from the honest grouped model. I display only safe numeric features, target, prediction probability, and error type — no client names, URLs, queries, or raw identifiers.


In [9]:
# Leakage checklist + deliberate leaky-feature test + real grouped-split failures.

FORBIDDEN_FEATURES = {
    "client_hash_id",
    "content_hash_id",
    "second_half_impressions",
    "future_trend_pct",
    "decline_proxy",
    "trend_pct",
    "trend_direction",
    "is_declining_label",
}

audit_rows = []

intersection = sorted(set(FEATURE_COLS) & FORBIDDEN_FEATURES)
audit_rows.append({
    "check": "No label-derived/future/ID fields in final features",
    "status": "PASS" if not intersection else "FAIL",
    "detail": "none" if not intersection else ", ".join(intersection)
})

audit_rows.append({
    "check": "Feature window ends before label window",
    "status": "PASS",
    "detail": "features Mar 1–15; label Mar 16–31"
})

audit_rows.append({
    "check": "Grouped split has zero client overlap",
    "status": "PASS" if len(train_clients & test_clients) == 0 else "FAIL",
    "detail": f"{len(train_clients & test_clients)} overlapping clients"
})

audit_rows.append({
    "check": "IDs used as predictors",
    "status": "PASS" if GROUP not in FEATURE_COLS and "content_hash_id" not in FEATURE_COLS else "FAIL",
    "detail": "IDs are grouping/context only"
})

leakage_audit = pd.DataFrame(audit_rows)
display(leakage_audit)

# Deliberately leaky grouped model — audit only.
leaky_cols = FEATURE_COLS + ["future_trend_pct"]
leaky_model = make_model()
leaky_model.fit(
    model_df.iloc[grp_train_idx][leaky_cols],
    y.iloc[grp_train_idx]
)
leaky_prob = leaky_model.predict_proba(
    model_df.iloc[grp_test_idx][leaky_cols]
)[:, 1]
leaky_pred = (leaky_prob >= 0.50).astype(int)
leaky_metrics = evaluate(y.iloc[grp_test_idx], leaky_prob, leaky_pred)

leak_demo = pd.DataFrame([
    {
        "model": "Honest grouped model",
        "precision_at_50": group_metrics["precision_at_50"],
        "pr_auc": group_metrics["pr_auc"],
        "roc_auc": group_metrics["roc_auc"],
    },
    {
        "model": "Audit-only model WITH future_trend_pct",
        "precision_at_50": leaky_metrics["precision_at_50"],
        "pr_auc": leaky_metrics["pr_auc"],
        "roc_auc": leaky_metrics["roc_auc"],
    },
]).set_index("model")

print("Leakage sanity check — the second row is intentionally invalid:")
display(leak_demo.round(3))

# Real failure examples from the honest grouped test set.
error_df = model_df.iloc[grp_test_idx][FEATURE_COLS + [TARGET]].copy()
error_df["prob_decline"] = group_prob
error_df["pred_decline"] = group_pred

error_df["error_type"] = np.select(
    [
        (error_df[TARGET] == 0) & (error_df["pred_decline"] == 1),
        (error_df[TARGET] == 1) & (error_df["pred_decline"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

false_pos = (
    error_df[error_df["error_type"] == "false_positive"]
    .sort_values("prob_decline", ascending=False)
    .head(5)
)

false_neg = (
    error_df[error_df["error_type"] == "false_negative"]
    .sort_values("prob_decline", ascending=True)
    .head(5)
)

print("Most confident false positives (safe numeric features only):")
display(false_pos.round(3))

print("Most confident false negatives (safe numeric features only):")
display(false_neg.round(3))

print("Confusion matrix for honest grouped model [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y.iloc[grp_test_idx], group_pred))

print(
    "\nError interpretation: false positives are pages whose first-half signals looked "
    "decline-like to the model but did not meet the outcome proxy; false negatives are "
    "observed proxy declines that the five pre-decision features did not rank strongly. "
    "These examples show limits of the signal set, not causes of search movement."
)


,check,status,detail
0,No label-derived/future/ID fields in final fea...,PASS,none
1,Feature window ends before label window,PASS,features Mar 1–15; label Mar 16–31
2,Grouped split has zero client overlap,PASS,0 overlapping clients
3,IDs used as predictors,PASS,IDs are grouping/context only


Leakage sanity check — the second row is intentionally invalid:


,precision_at_50,pr_auc,roc_auc
model,,,
Honest grouped model,0.34,0.395,0.549
Audit-only model WITH future_trend_pct,1.00,1.000,1.000


Most confident false positives (safe numeric features only):


,first_half_impressions,first_half_clicks,first_half_avg_position,first_half_active_days,first_half_ctr_pct,decline_proxy,prob_decline,pred_decline,error_type
87621,25042.0,7.0,38.271,15,0.028,0,0.889,1,false_positive
147968,1.0,1.0,9.000,1,100.000,0,0.715,1,false_positive
143686,1.0,1.0,9.000,1,100.000,0,0.715,1,false_positive
147392,1.0,1.0,11.000,1,100.000,0,0.713,1,false_positive
88349,19059.0,19.0,30.666,15,0.100,0,0.709,1,false_positive


Most confident false negatives (safe numeric features only):


,first_half_impressions,first_half_clicks,first_half_avg_position,first_half_active_days,first_half_ctr_pct,decline_proxy,prob_decline,pred_decline,error_type
12835,2678.0,77.0,1.798,15,2.875,1,0.029,0,false_negative
38609,5603.0,77.0,1.630,15,1.374,1,0.039,0,false_negative
12167,14244.0,95.0,3.214,15,0.667,1,0.041,0,false_negative
138580,3338.0,55.0,1.242,11,1.648,1,0.098,0,false_negative
12861,3366.0,47.0,1.323,15,1.396,1,0.120,0,false_negative


Confusion matrix for honest grouped model [[TN, FP], [FN, TP]]:
[[3973 4305]
 [2094 2838]]

Error interpretation: false positives are pages whose first-half signals looked decline-like to the model but did not meet the outcome proxy; false negatives are observed proxy declines that the five pre-decision features did not rank strongly. These examples show limits of the signal set, not causes of search movement.


## 4. Claim rewrite

A sentence I would **not** use publicly is:

> **“My model predicts which pages will decline.”**

That wording is too strong for this design. The target is a **proxy** defined from a single March before/after impression comparison, and even the honest test is one client-grouped holdout rather than evidence across many future months.

My public-safe version is:

> **“On a client-held-out March 2026 evaluation, a Logistic Regression model using five pre-decision Search Console features produced a measured ranking of pages associated with the defined decline proxy. I treat that score as directional decision-support for prioritizing human review, not as a prediction of Google’s ranking algorithm or proof of why a page declined.”**

I would add the actual grouped Precision@50 only after the cell below measures it. If the grouped model does not beat the relevant baseline or base rate, I will say that directly rather than preserving a stronger Week-5 story.

The most important change is not cosmetic wording: the revised claim names the **population, validation design, proxy target, and intended use**, and it separates an observed predictive association from a causal explanation.


In [10]:
# Generate the measured public-safe claim from the honest grouped result.
# This prevents me from hard-coding a flattering number before the validation run.

group_p50 = group_metrics["precision_at_50"]
group_base = group_metrics["base_rate"]
random_p50 = random_metrics["precision_at_50"]

safe_claim = (
    f"On the client-held-out March 2026 evaluation, the Logistic Regression model "
    f"measured Precision@50 = {group_p50:.3f} for the defined decline proxy "
    f"(held-out base rate = {group_base:.3f}). "
    f"The naive random-split Precision@50 was {random_p50:.3f}. "
    "I treat the grouped result as directional decision-support for prioritizing human review, "
    "not as evidence that the model predicts Google's ranking algorithm or identifies the cause of decline."
)

print(safe_claim)

if group_p50 < group_base:
    print(
        "\nClaim correction: on this grouped split, the model's top-50 precision is below "
        "the held-out base rate. I would not claim useful ranking lift from this run."
    )
else:
    print(
        "\nClaim boundary: the measured ranking may be useful on this held-out split, "
        "but broader generalization still needs repeated grouped and/or future-time validation."
    )


On the client-held-out March 2026 evaluation, the Logistic Regression model measured Precision@50 = 0.340 for the defined decline proxy (held-out base rate = 0.373). The naive random-split Precision@50 was 0.620. I treat the grouped result as directional decision-support for prioritizing human review, not as evidence that the model predicts Google's ranking algorithm or identifies the cause of decline.

Claim correction: on this grouped split, the model's top-50 precision is below the held-out base rate. I would not claim useful ranking lift from this run.


## Self-check

Before I submit, I confirm each line honestly:

- [x] Every required section is filled with markdown reasoning and executable code.
- [x] Two findings from the FlyRank paper are named and reviewed constructively.
- [x] The model code shows a naive random split **before** and a client-grouped split **after**.
- [x] The grouped split checks that no client appears in both train and test.
- [x] The final feature set uses only March 1–15 information; the label uses March 16–31.
- [x] Label-derived, future-window, ID, and product-decision leakage paths are explicitly audited.
- [x] An intentionally leaky feature is tested only as an audit sanity check and is never accepted as a production feature.
- [x] Real false-positive and false-negative examples are displayed without client names, URLs, private queries, or raw IDs.
- [x] Public claims use careful language: **observed, measured, directional, decision-support**.
- [ ] Run the notebook top to bottom in Colab with my private `HF_TOKEN` and confirm all output cells are visible.
- [ ] After the successful run, save/commit it as `work/notebooks/w06_validation_audit.ipynb`.
- [ ] Submit the repository URL on the assignment card.

**Submission rule:** I will not check the last three boxes until the gated warehouse query has actually run and the executed notebook is saved in GitHub.
